In [3]:
%pip install fastapi uvicorn scikit-learn pandas numpy mlflow jupyter

Defaulting to user installation because normal site-packages is not writeable
  Using cached fastapi-0.135.1-py3-none-any.whl (116 kB)
  Using cached uvicorn-0.42.0-py3-none-any.whl (68 kB)
  Using cached mlflow-3.10.1-py3-none-any.whl (10.2 MB)
  Using cached jupyter-1.1.1-py2.py3-none-any.whl (2.7 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl (5.3 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
  Using cached starlette-0.52.1-py3-none-any.whl (74 kB)
  Using cached flask-3.1.3-py3-none-any.whl (103 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl (114 kB)
  Using cached alembic-1.18.4-py3-none-any.whl (263 kB)
  Using cached skops-0.13.0-py3-none-any.whl (131 kB)
  Using cached flask_cors-6.0.2-py3-none-any.whl (13 kB)
  Using cached mlflow_tracing-3.10.1-py3-none-any.whl (1.5 MB)
  Using cached mlflow_skinny-3.10.1-py3-none-any.whl (3.0 MB)
  Using cached gunicorn-25.1.0-py3-none-any.whl (19

In [1]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd

In [6]:

mlflow.set_tracking_uri("http://localhost:5000")

data = load_diabetes(scaled=False)
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [7]:

experiment_name = "diabetes_experiment"
try:
    experiment_id = mlflow.create_experiment(experiment_name)
except:
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

with mlflow.start_run(experiment_id=experiment_id, run_name="random_forest_with_scaler"):
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestRegressor(n_estimators=100, random_state=42))
    ])
    
    pipeline.fit(X_train, y_train)
    
    train_score = pipeline.score(X_train, y_train)
    test_score = pipeline.score(X_test, y_test)
    
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("random_state", 42)
    mlflow.log_metric("train_r2", train_score)
    mlflow.log_metric("test_r2", test_score)
    
    mlflow.sklearn.log_model(pipeline, "model")
    
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"
    mlflow.register_model(model_uri, "diabetes_model")
    
    print(f"Run ID: {mlflow.active_run().info.run_id}")
    print("Модель зарегистрирована как 'diabetes_model'")

2026/03/17 19:06:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/17 19:06:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'diabetes_model' already exists. Creating a new version of this model...
2026/03/17 19:06:45 WARNING mlflow.tracking._model_registry.fluent: Run with id 51a59c76e4984e5292b5d8ec984aef11 has no artifacts at artifact path 'model', registering model based on models:/m-62064b0a9a60445b8b842d041a3e1eee instead
2026/03/17 19:06:45 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: diabetes_model, version

Run ID: 51a59c76e4984e5292b5d8ec984aef11
Модель зарегистрирована как 'diabetes_model'
🏃 View run random_forest_with_scaler at: http://localhost:5000/#/experiments/348208227634853730/runs/51a59c76e4984e5292b5d8ec984aef11
🧪 View experiment at: http://localhost:5000/#/experiments/348208227634853730
